In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


Data Reading


In [0]:
df = spark.read.format("parquet").load("abfss://bronze@datalaketanish.dfs.core.windows.net/customers")

In [0]:
df = df.drop("_rescued_data")
df.limit(5).display()

customer_id,first_name,last_name,email,city,state
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI


In [0]:
df = df.withColumn("domains",split(col("email"), "@")[1])
df.limit(5).display()

customer_id,first_name,last_name,email,city,state,domains
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY,kelly.com
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND,campbell.info
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,yahoo.com


In [0]:
df.groupby("domains").agg(count("customer_id").alias("total_customers")).sort("total_customers",ascending=False).limit(5).display()

domains,total_customers
gmail.com,374
hotmail.com,360
yahoo.com,331
davis.com,8
brown.com,8


In [0]:
df_gmail = df.filter(col('domains') == "gmail.com")
df_yahoo = df.filter(col('domains') == "yahoo.com")
df_hotmail = df.filter(col('domains') == "hotmail.com")
df_gmail.limit(5).display()
df_yahoo.limit(5).display()
df_hotmail.limit(5).display()


customer_id,first_name,last_name,email,city,state,domains
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,gmail.com
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,gmail.com
C00011,Jacob,Le,anita65@gmail.com,Houstonfurt,AR,gmail.com
C00012,Chad,Banks,beardtravis@gmail.com,East Jenniferview,WI,gmail.com
C00013,James,Martin,jwood@gmail.com,Lake Gregoryshire,OR,gmail.com


customer_id,first_name,last_name,email,city,state,domains
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,yahoo.com
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com
C00014,Thomas,Hartman,lmoore@yahoo.com,East Tiffanybury,MS,yahoo.com
C00024,Heather,Owens,andreawilliams@yahoo.com,Stewartport,NM,yahoo.com


customer_id,first_name,last_name,email,city,state,domains
C00015,Diane,Harris,daniellowe@hotmail.com,South Courtney,SC,hotmail.com
C00020,Juan,Collins,rojassandra@hotmail.com,South Oscar,NV,hotmail.com
C00031,Jeanette,Smith,kramerkaylee@hotmail.com,Travisview,CT,hotmail.com
C00033,Carrie,Wheeler,lindasummers@hotmail.com,South Andrewchester,CO,hotmail.com
C00036,Jacob,Mccoy,jesustaylor@hotmail.com,Port Nicoleborough,AR,hotmail.com


In [0]:
df = df.withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name")))
df = df.drop('first_name','last_name')
df.limit(5).display()


customer_id,email,city,state,domains,full_name
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez


In [0]:
df.write.mode("overwrite").format("delta").save("abfss://silver@datalaketanish.dfs.core.windows.net/customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  databricks_cata.silver.customers_silver 
USING DELTA
LOCATION 'abfss://silver@datalaketanish.dfs.core.windows.net/customers'